In [5]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("bryanpark/sudoku")

print("Path to dataset files:", path)

Path to dataset files: C:\Users\anajr\.cache\kagglehub\datasets\bryanpark\sudoku\versions\3


In [6]:
import copy
import sudoku
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader


solution = sudoku.construct_puzzle_solution()
puzzle, givens = sudoku.pluck(copy.deepcopy(solution), n=30)
print(puzzle)
sudoku.display(puzzle)
print("givens:", givens)

[[0, 0, 8, 3, 0, 4, 0, 7, 0], [0, 9, 0, 0, 6, 1, 5, 0, 8], [0, 0, 0, 0, 9, 0, 6, 2, 4], [7, 0, 4, 0, 8, 5, 0, 6, 3], [3, 0, 0, 0, 1, 2, 0, 5, 9], [1, 0, 0, 0, 0, 3, 4, 0, 0], [6, 1, 2, 0, 0, 8, 0, 9, 7], [0, 0, 0, 7, 0, 0, 0, 1, 0], [8, 0, 5, 0, 0, 0, 2, 0, 6]]
_ _ 8 3 _ 4 _ 7 _
_ 9 _ _ 6 1 5 _ 8
_ _ _ _ 9 _ 6 2 4
7 _ 4 _ 8 5 _ 6 3
3 _ _ _ 1 2 _ 5 9
1 _ _ _ _ 3 4 _ _
6 1 2 _ _ 8 _ 9 7
_ _ _ 7 _ _ _ 1 _
8 _ 5 _ _ _ 2 _ 6
givens: 39


In [7]:
import numpy as np
quizzes = np.zeros((1000000, 81), np.int32)
solutions = np.zeros((1000000, 81), np.int32)
for i, line in enumerate(open('data/sudoku.csv', 'r').read().splitlines()[1:]):
    quiz, solution = line.split(",")
    for j, q_s in enumerate(zip(quiz, solution)):
        q, s = q_s
        quizzes[i, j] = q
        solutions[i, j] = s
quizzes = quizzes.reshape((-1, 9, 9))
solutions = solutions.reshape((-1, 9, 9))

In [8]:
print(quizzes)

[[[0 0 4 ... 2 0 9]
  [0 0 5 ... 0 0 1]
  [0 7 0 ... 0 4 3]
  ...
  [6 0 0 ... 1 0 5]
  [0 0 3 ... 6 9 0]
  [0 4 2 ... 3 0 0]]

 [[0 4 0 ... 0 5 0]
  [1 0 7 ... 9 6 0]
  [5 2 0 ... 0 0 0]
  ...
  [0 9 0 ... 5 4 3]
  [6 0 0 ... 7 0 0]
  [2 5 0 ... 1 0 0]]

 [[6 0 0 ... 3 8 4]
  [0 0 8 ... 0 7 2]
  [0 0 0 ... 0 0 5]
  ...
  [3 1 0 ... 0 5 0]
  [0 8 9 ... 0 0 0]
  [5 0 2 ... 1 9 0]]

 ...

 [[0 0 0 ... 8 2 0]
  [0 6 1 ... 0 3 0]
  [0 5 0 ... 0 0 0]
  ...
  [0 0 7 ... 0 6 5]
  [0 0 0 ... 4 0 8]
  [0 8 6 ... 0 0 0]]

 [[0 7 0 ... 6 9 0]
  [0 0 3 ... 0 0 1]
  [0 0 0 ... 0 2 0]
  ...
  [0 0 0 ... 0 4 0]
  [0 5 1 ... 9 0 0]
  [9 4 0 ... 0 0 7]]

 [[3 0 0 ... 6 2 0]
  [1 0 0 ... 4 0 0]
  [0 0 5 ... 8 3 0]
  ...
  [4 8 0 ... 0 1 0]
  [2 0 3 ... 0 0 0]
  [0 7 0 ... 0 9 0]]]


In [9]:
df = pd.read_csv('data/sudoku.csv',dtype=str)
df

,quizzes,solutions
0,0043002090050090010700600430060020871900074000...,8643712593258497619712658434361925871986574322...
1,0401000501070039605200080000000000170009068008...,3461792581875239645296483719658324174729168358...
2,6001203840084590720000060050002640300700800069...,6951273841384596727248369158512647392739815469...
3,4972000001004000050000160986203000403009000000...,4972583161864397252537164986293815473759641828...
4,0059103080094030600275001000300002010008200070...,4659123781894735623275681497386452919548216372...
...,...,...
999995,3000280000290000300054001077402030980086070031...,3175289464291768356854391277462135989586472131...
999996,0030006000040860057000009409350407208067200502...,5234976811942863757685139429356417288167294532...
999997,0003508200618040300500090000700600029030070100...,7493568212618745393582197468749613529235876146...
999998,0702006900030400010000650205600300000947005800...,4752816936239478511893657245628341793947165828...


In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 2 columns):
 #   Column     Non-Null Count    Dtype 
---  ------     --------------    ----- 
 0   quizzes    1000000 non-null  object
 1   solutions  1000000 non-null  object
dtypes: object(2)
memory usage: 15.3+ MB


In [11]:
# 81자리가 아닌 문제 확인 > 없음
df[df['quizzes'].str.len()!=81]

,quizzes,solutions


In [12]:
# 81자리가 아닌 답안지 확인 > 없음
df[df['solutions'].str.len()!=81]

,quizzes,solutions


입력(quiz) :  0~9 (0은 빈칸을 뜻함)  
정답(solution) : 1~9  
학습 라벨 : (solution - 1) → 0~8

In [13]:
quiz_str = df['quizzes'][0]
quiz = np.array([int(c) for c in quiz_str]).reshape(9,9).astype(np.int64)
quiz

array([[0, 0, 4, 3, 0, 0, 2, 0, 9],
       [0, 0, 5, 0, 0, 9, 0, 0, 1],
       [0, 7, 0, 0, 6, 0, 0, 4, 3],
       [0, 0, 6, 0, 0, 2, 0, 8, 7],
       [1, 9, 0, 0, 0, 7, 4, 0, 0],
       [0, 5, 0, 0, 8, 3, 0, 0, 0],
       [6, 0, 0, 0, 0, 0, 1, 0, 5],
       [0, 0, 3, 5, 0, 8, 6, 9, 0],
       [0, 4, 2, 9, 1, 0, 3, 0, 0]])

In [14]:


class SudokuDataset(Dataset):
    """
    반환:
      X: (10, 9, 9) float32 one-hot (0~9)
      y: (9, 9) int64  (0~8)  # 정답 1~9를 0~8로 shift
      mask: (9, 9) bool  # quiz==0 (빈칸 위치)
    """
    def __init__(self, df: pd.DataFrame, quiz_col: str, sol_col: str):
        self.quiz = df[quiz_col].values
        self.sol  = df[sol_col].values

    def __len__(self):
        return len(self.quiz)

    @staticmethod
    def _str_to_grid(s: str) -> np.ndarray:

        b = np.frombuffer(s.encode("ascii"), dtype=np.uint8) - ord("0")
        return b.reshape(9, 9).astype(np.int64)

    def __getitem__(self, idx: int):
        quiz_str = self.quiz[idx]
        sol_str  = self.sol[idx]

        quiz = self._str_to_grid(quiz_str)     # (9,9) 0~9
        sol  = self._str_to_grid(sol_str)      # (9,9) 1~9

        mask = (quiz == 0)                                  # 마스크: 빈칸 위치만 학습,평가에 쓰는 용도

        # 라벨: 1~9 -> 0~8
        y = sol - 1

        # 입력 원핫: 0~9 (10클래스)
        # X[d, i, j] = 1 if quiz[i,j] == d
        X = np.zeros((10, 9, 9), dtype=np.float32)
        for d in range(10):
            X[d] = (quiz == d)

        # torch 텐서로 변환
        X = torch.from_numpy(X)                 # float32
        y = torch.from_numpy(y.astype(np.int64))# int64
        mask = torch.from_numpy(mask)           # bool

        return X, y, mask

In [15]:
dataset = SudokuDataset(df, "quizzes", "solutions")

loader = DataLoader(
    dataset,
    batch_size=64,
    shuffle=True,
    num_workers=0,      # 윈도우면 0~2부터 시작 추천
    pin_memory=True
)

X, y, mask = next(iter(loader))
print("X:", X.shape, X.dtype)           # (B, 10, 9, 9) torch.float32
print("y:", y.shape, y.dtype)           # (B, 9, 9) torch.int64
print("mask:", mask.shape, mask.dtype)  # (B, 9, 9) torch.bool
print("blank ratio in batch:", mask.float().mean().item())

X: torch.Size([64, 10, 9, 9]) torch.float32
y: torch.Size([64, 9, 9]) torch.int64
mask: torch.Size([64, 9, 9]) torch.bool
blank ratio in batch: 0.5790895223617554


c:\python_workspace\python\deep_learning\dl_venv\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


In [16]:
class SudokuCNN(nn.Module):
    def __init__(self, in_ch=10, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(X[0].size(0), hidden, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(hidden, hidden, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(hidden, hidden, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(hidden, 9, 1)  # 각 칸 9클래스 logits
        )

    def forward(self, x):
        logits = self.net(x)
        return logits

## conv2d  
작은 필터(예: 3×3)를  
전체 격자에 공유해서  
이웃 패턴을 인식  
주변 칸을 보면서 규칙을 학습하는 연산


In [17]:
def masked_ce_loss(logits, y, mask):
    B = y.size(0)
    # (B,C,H,W) -> (B*H*W, C)
    logits = logits.permute(0, 2, 3, 1).reshape(B*81, 9)
    y = y.reshape(B*81)
    mask = mask.reshape(B*81)

    logits_m = logits[mask]
    y_m = y[mask]

    # 빈칸이 하나도 없는 배치면(거의 없지만) 안전 처리
    if logits_m.numel() == 0:
        return torch.tensor(0.0, device=logits.device, requires_grad=True)

    return F.cross_entropy(logits_m, y_m)

In [18]:
import time

device = torch.device("cpu")

model = SudokuCNN(in_ch=10, hidden=64).to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

model.train()
for epoch in range(1, 4):  # 일단 3epoch만
    t0 = time.time()
    total_loss = 0.0
    total_cells = 0
    correct_cells = 0

    for X, y, mask in loader:
        X = X.to(device)
        y = y.to(device)
        mask = mask.to(device)

        opt.zero_grad()
        logits = model(X)  # (B, 9, 9, 9) (B,C,H,W)

        loss = masked_ce_loss(logits, y, mask)
        loss.backward()
        opt.step()

        total_loss += loss.item()

        # 빈칸 cell accuracy (모니터링용)
        with torch.no_grad():
            pred = logits.argmax(dim=1)  # (B, 9, 9)
            m = mask
            total_cells += m.sum().item()
            correct_cells += ((pred == y) & m).sum().item()

    dt = time.time() - t0
    acc = (correct_cells / total_cells) if total_cells > 0 else 0.0
    print(f"epoch {epoch} | loss {total_loss/len(loader):.4f} | blank_acc {acc:.4f} | time {dt:.1f}s")


c:\python_workspace\python\deep_learning\dl_venv\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


epoch 1 | loss 1.5857 | blank_acc 0.2910 | time 307.4s
epoch 2 | loss 1.5637 | blank_acc 0.2964 | time 298.7s
epoch 3 | loss 1.5535 | blank_acc 0.3006 | time 328.0s


In [36]:
import numpy as np
import torch
import copy

solution = sudoku.construct_puzzle_solution()
puzzle, givens = sudoku.pluck(copy.deepcopy(solution), n=30)

quiz = np.array(puzzle, dtype=np.int64)  # (9,9)
mask = (quiz == 0)

# one-hot (10,9,9)
X = np.zeros((10, 9, 9), dtype=np.float32)
for d in range(10):
    X[d] = (quiz == d)

# torch 텐서: (1,10,9,9)
a = torch.from_numpy(X).unsqueeze(0)  # add batch dim

model.eval()
with torch.no_grad():
    logits = model(a)                 # (1,9,9,9) = (B,C,H,W)
    pred = logits.argmax(dim=1)[0]    # (9,9) 0~8

pred_1to9 = (pred + 1).cpu().numpy()  # (9,9)

filled = quiz.copy()
filled[mask] = pred_1to9[mask]

print("QUIZ:")
print(quiz)
print("\nFILLED:")
print(filled)


QUIZ:
[[1 0 0 3 6 0 8 0 9]
 [2 0 7 0 1 0 0 0 0]
 [0 0 0 4 0 0 7 3 0]
 [5 0 9 1 0 4 3 7 0]
 [0 0 0 6 0 0 0 1 0]
 [3 0 0 0 8 0 6 0 5]
 [0 8 2 5 0 3 0 9 6]
 [9 0 5 0 7 0 0 0 0]
 [0 4 0 8 0 0 2 0 0]]

FILLED:
[[1 9 5 3 6 2 8 4 9]
 [2 6 7 8 1 9 4 6 1]
 [6 6 8 4 2 9 7 3 6]
 [5 2 9 1 2 4 3 7 2]
 [7 2 4 6 3 9 4 1 4]
 [3 7 4 9 8 2 6 2 5]
 [6 8 2 5 1 3 4 9 6]
 [9 6 5 1 7 6 5 3 1]
 [2 4 3 8 9 6 2 7 3]]


In [40]:
import numpy as np
import torch

def quiz_to_X(quiz):
    X = np.zeros((10, 9, 9), dtype=np.float32)
    for d in range(10):
        X[d] = (quiz == d)
    return torch.from_numpy(X).unsqueeze(0)  # (1,10,9,9)

def valid_candidates(board, r, c):
    """현재 보드 기준으로 (r,c)에 들어갈 수 있는 후보(1~9) 반환"""
    if board[r, c] != 0:
        return set()

    used = set(board[r, :]) | set(board[:, c])
    br, bc = (r//3)*3, (c//3)*3
    used |= set(board[br:br+3, bc:bc+3].reshape(-1))
    used.discard(0)
    return set(range(1,10)) - used

def iterative_fill(model, quiz, max_steps=81, topk=1, device="cpu"):
    """
    topk=1이면 매 스텝에서 가장 확신 높은 1칸만 채움(가장 안전)
    topk=3~5면 조금 더 빠르게 채움(충돌 가능성 조금 증가)
    """
    board = quiz.copy().astype(np.int64)
    model.eval()

    for step in range(max_steps):
        blanks = np.argwhere(board == 0)
        if len(blanks) == 0:
            break

        X = quiz_to_X(board).to(device)

        with torch.no_grad():
            logits = model(X)  # (1,9,9,9) (B,C,H,W)
            # 확률로 변환: (9,9,9)
            probs = torch.softmax(logits[0].permute(1,2,0), dim=-1).cpu().numpy()

        # 각 빈칸에 대해 "규칙을 만족하는 후보 중에서" 가장 높은 확률을 선택
        scored = []
        for r, c in blanks:
            cand = valid_candidates(board, r, c)
            if not cand:
                continue
            cand_list = sorted(cand)
            p = probs[r, c, np.array(cand_list)-1]   # 클래스 0~8이 1~9에 대응
            best_i = int(np.argmax(p))
            best_val = cand_list[best_i]
            best_conf = float(p[best_i])
            scored.append((best_conf, r, c, best_val))

        if not scored:
            # 더 이상 규칙을 만족하며 채울 수 있는 칸이 없음
            break

        scored.sort(reverse=True)  # 확신 높은 순
        for conf, r, c, v in scored[:topk]:
            board[r, c] = v

    return board

def show(board):
    for r in board:
        print(" ".join(str(int(x)) if x != 0 else "_" for x in r))

# 사용 예시:
filled_iter = iterative_fill(model, quiz, topk=1)
print("=== ITER FILLED ===")
show(filled_iter)


=== ITER FILLED ===
1 5 _ 3 6 2 8 _ 9
2 9 7 _ 1 5 4 6 _
6 _ _ 4 9 8 7 3 2
5 6 9 1 2 4 3 7 8
7 2 8 6 5 _ 9 1 4
3 1 4 7 8 9 6 2 5
_ 8 2 5 4 3 _ 9 6
9 _ 5 2 7 6 _ 4 1
_ 4 3 8 _ 1 2 5 7
